## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
# Create DataFrame from SQL query
df = spark.sql("""
SELECT *
FROM workspace.default.sales_december_2019
""")

df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|         

Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
# Your code here# Create widget parameter
dbutils.widgets.text("Product_name", "", "Product_name")

# Read parameter value
product_name = dbutils.widgets.get("Product_name")

# Get filtered dataframe using PySpark
filtered_df = df.filter(df["Product"] == product_name)

# Show data
filtered_df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|               2|     11.95|12/13/19 09:29|813 Hickory St, D...|
|  295675|USB-C Charging Cable|               2|     11.95|12/13/19 13:52|594 1st St, San F...|
|  295679|USB-C Charging Cable|               1|     11.95|12/25/19 09:39|902 2nd St, Dalla...|
|  295681|USB-C Charging Cable|               1|     11.95|12/25/19 12:37|79 Elm St, Boston...|
|  295682|USB-C Charging Cable|         

As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
# Your code here
from pyspark.sql.functions import col, sum


df_clean = df.filter(
    col("Quantity Ordered") != "Quantity Ordered"
)


df_casted = df_clean.withColumn(
    "Quantity Ordered", col("Quantity Ordered").cast("int")
).withColumn(
    "Price Each", col("Price Each").cast("double")
)

agg_df = df_casted.groupBy("Product").agg(
    sum("Quantity Ordered").alias("total_quantity"),
    sum(col("Quantity Ordered") * col("Price Each")).alias("total_sales")
)

agg_df.show()

+--------------------+--------------+------------------+
|             Product|total_quantity|       total_sales|
+--------------------+--------------+------------------+
|  Macbook Pro Laptop|           644|         1094800.0|
|  LG Washing Machine|            80|           48000.0|
|USB-C Charging Cable|          3251| 38849.45000000006|
|    27in FHD Monitor|           965| 144740.3500000011|
|AA Batteries (4-p...|          3718|14277.120000000394|
|Bose SoundSport H...|          1825|182481.74999999825|
|AAA Batteries (4-...|          4240|12677.599999999413|
|     ThinkPad Laptop|           541| 540994.5899999965|
|Lightning Chargin...|          3089| 46180.54999999874|
|        Google Phone|           716|          429600.0|
|    Wired Headphones|          2748| 32948.52000000157|
|Apple Airpods Hea...|          2079|          311850.0|
|     Vareebadd Phone|           285|          114000.0|
|              iPhone|           908|          635600.0|
|        20in Monitor|         

In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
# Your code here
from pyspark.sql.functions import col

df_clean = df.filter(col("Quantity Ordered").rlike("^[0-9]+$"))

df_casted = df_clean.withColumn(
    "Quantity Ordered", col("Quantity Ordered").cast("int")
).withColumn(
    "Price Each", col("Price Each").cast("double")
)

print("Row count:", df_casted.count())

df_casted.summary().show()

Row count: 24989
+-------+-----------------+------------+-------------------+------------------+--------------+--------------------+
|summary|         Order ID|     Product|   Quantity Ordered|        Price Each|    Order Date|    Purchase Address|
+-------+-----------------+------------+-------------------+------------------+--------------+--------------------+
|  count|            24989|       24989|              24989|             24989|         24989|               24989|
|   mean|307655.0231701949|        NULL| 1.1253351474648845|183.84565008610213|          NULL|                NULL|
| stddev|6932.795455986289|        NULL|0.44541356230083035|333.07703685801806|          NULL|                NULL|
|    min|           295665|20in Monitor|                  1|              2.99|01/01/20 00:10|1 12th St, San Fr...|
|    25%|         301652.0|        NULL|                  1|             11.95|          NULL|                NULL|
|    50%|         307654.0|        NULL|               


5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
#your code here
from pyspark.sql.functions import lit

df_new = df_casted.withColumn("source", lit("online"))

df_new.show()

+--------+--------------------+----------------+----------+--------------+--------------------+------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|source|
+--------+--------------------+----------------+----------+--------------+--------------------+------+
|  295665|  Macbook Pro Laptop|               1|    1700.0|12/30/19 00:01|136 Church St, Ne...|online|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|online|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|online|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|online|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|online|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|online|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10

Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
# Your code here for view creation
df_new.createOrReplaceTempView("sales_view")


In [0]:

%sql
SELECT 
    Product, 
    SUM(`Quantity Ordered`) AS total_quantity
FROM sales_view
GROUP BY Product
ORDER BY total_quantity DESC

Product,total_quantity
AAA Batteries (4-pack),4240
AA Batteries (4-pack),3718
USB-C Charging Cable,3251
Lightning Charging Cable,3089
Wired Headphones,2748
Apple Airpods Headphones,2079
Bose SoundSport Headphones,1825
27in FHD Monitor,965
iPhone,908
27in 4K Gaming Monitor,861
